In [ ]:
!pip install --no-index --find-links /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels arc-agi python-dotenv

In [ ]:
import os
import shutil
import sys
import traceback

print("=== UNCONDITIONAL DIAGNOSTIC v2 (mirrors the real submission notebook's exact setup order) ===", flush=True)

import numpy
print(f"python {sys.version}", flush=True)
print(f"numpy {numpy.__version__}", flush=True)

print("\n--- step: copy competition harness repo ---", flush=True)
shutil.copytree(
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents",
    "/kaggle/working/ARC-AGI-3-Agents",
    ignore=shutil.ignore_patterns(".git"),
)
print("harness copy OK", flush=True)

print("\n--- step: copy graph-explorer agent files from dataset ---", flush=True)
_dataset_root = "/kaggle/input/datasets/calamitychasm/graph-explorer-agent"
_templates_dir = "/kaggle/working/ARC-AGI-3-Agents/agents/templates"
shutil.copy(f"{_dataset_root}/graph_explorer_core.py", f"{_templates_dir}/graph_explorer_core.py")
shutil.copy(f"{_dataset_root}/graph_explorer_agent.py", f"{_templates_dir}/graph_explorer_agent.py")
for p in [f"{_templates_dir}/graph_explorer_core.py", f"{_templates_dir}/graph_explorer_agent.py"]:
    assert os.path.exists(p), f"missing: {p}"
print("agent copy OK", flush=True)

print("\n--- step: write minimal agents/__init__.py (EXACTLY matching the real submission notebook) ---", flush=True)
with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
    f.write(
        "from typing import Type, cast\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.graph_explorer_agent import GraphExplorerAgent\n"
        "\n"
        "load_dotenv()\n"
        "\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n"
        "    \"random\": Random,\n"
        "    \"graphexploreragent\": GraphExplorerAgent,\n"
        "}\n"
    )
print("wrote minimal __init__.py", flush=True)

print("\n--- step: sanity-import the agent (same path main.py itself will use) ---", flush=True)
try:
    sys.path.insert(0, "/kaggle/working/ARC-AGI-3-Agents")
    sys.path.insert(0, "/kaggle/working")
    from agents import AVAILABLE_AGENTS
    print(f"agents import OK: {list(AVAILABLE_AGENTS.keys())}", flush=True)
    from agents.templates.graph_explorer_agent import GraphExplorerAgent, FrameProcessor
    print("GraphExplorerAgent/FrameProcessor import OK", flush=True)
except Exception:
    print("IMPORT FAILED:", flush=True)
    traceback.print_exc()
    raise SystemExit(0)

print("\n--- step: run FrameProcessor on a synthetic frame (the actual thing that runs on EVERY real decision) ---", flush=True)
try:
    import numpy as np
    fp = FrameProcessor()
    frame = np.zeros((64, 64), dtype=np.uint8)
    frame[0:3, :] = 5
    frame[10:15, 10:15] = 9
    frame[30:34, 40:50] = 12
    frame[50:52, 5:60] = 2

    segmented_frame, frame_segments = fp.segment_frame(frame)
    print(f"segment_frame OK: {len(frame_segments)} segments", flush=True)

    status_bar_segments_list, status_bar_mask = fp.identify_status_bars(segmented_frame, frame_segments)
    print(f"identify_status_bars OK: mask sum={status_bar_mask.sum()}", flush=True)

    action_groups = fp.frame_segments_to_action_groups(frame_segments, n_groups=5)
    print(f"frame_segments_to_action_groups OK: group sizes={[len(g) for g in action_groups]}", flush=True)

    h = fp.hash_frame(frame)
    print(f"hash_frame OK: {h}", flush=True)

    print("=== ALL FRAMEPROCESSOR DIAGNOSTICS PASSED ===", flush=True)
except Exception:
    print("FRAMEPROCESSOR DIAGNOSTIC FAILED:", flush=True)
    traceback.print_exc()

print("\n--- step: full end-to-end choose_action simulation using a fake Agent construction ---", flush=True)
try:
    from unittest.mock import MagicMock
    from arcengine import FrameData, GameState

    # Build a minimal fake agent instance bypassing the real network-backed
    # __init__ (no arc_env available in a diagnostic push) -- calls
    # GraphExplorerAgent.__init__ directly is not possible without arc_env,
    # so instead construct via __new__ and manually set the attributes
    # choose_action actually needs, then call the real method.
    agent = GraphExplorerAgent.__new__(GraphExplorerAgent)
    agent.game_id = "diag-game"
    agent.frame_processor = FrameProcessor()
    agent.status_bar_mask = None
    agent.hashed_frame2action_results = {}
    agent.hashed_frame2transitions = {}
    agent.last_hashed_frame = None
    agent.last_action = None
    agent.arrow_control = True
    agent.favor_new_actions = False
    agent.favor_frontier_search = True
    from agents.templates.graph_explorer_core import GraphExplorer
    agent.graph_explorer = GraphExplorer(verbose_level=0, n_groups=agent.N_GROUPS)
    agent.level_first_frame = None
    agent.failed = False
    agent.level_up = True
    from arcengine import GameAction
    agent.last_action_object = GameAction.RESET
    import time
    agent.time_start = time.time()
    agent.last_time = time.time()
    agent.last_transition_suspicious = False
    agent._prev_levels_completed = 0

    frame = np.zeros((64, 64), dtype=np.uint8)
    frame[0:3, :] = 5
    frame[20:25, 20:25] = 9
    frame[40:44, 10:20] = 12

    fake_frame = FrameData(
        game_id="diag-game",
        frame=[frame.tolist()],
        state=GameState.NOT_FINISHED,
        levels_completed=0,
        available_actions=[1, 2, 3, 6],
    )
    action = agent.choose_action([fake_frame], fake_frame)
    print(f"choose_action OK: returned {action}", flush=True)
    print("=== END-TO-END DIAGNOSTIC PASSED ===", flush=True)
except Exception:
    print("END-TO-END DIAGNOSTIC FAILED:", flush=True)
    traceback.print_exc()

print("\n=== DIAGNOSTIC COMPLETE ===", flush=True)


In [ ]:
import pandas as pd
submission = pd.DataFrame(
    data=[['1_0', '1', True, 1]],
    columns=['row_id', 'game_id', 'end_of_game', 'score'])
submission.to_parquet('/kaggle/working/submission.parquet', index=False)
